### 속성기반 감성분석 ABSA (국내 KR / 글로벌 GB)

## 개요

리뷰 문장을 절 단위로 분리하고, 각 절 안에서 **"속성 단어" 주변에 등장하는 감성어**를 찾아 점수를 매기는 방식의 ABSA(Aspect-Based Sentiment Analysis)입니다. TF-IDF/N-gram/연관규칙이 "어떤 단어가 많이 나오는가"를 봤다면, 이 분석은 **"각 속성을 소비자가 얼마나 긍정/부정적으로 평가하는가"** 를 -1~+1 사이 점수로 계산합니다.

### 처리 순서
1. 역접 접속사(하지만/그런데 등, but/however 등) 기준으로 리뷰를 절(clause) 단위로 분리
   → "발림성은 좋은데 백탁이 심해요" 같은 문장에서 두 속성이 서로 다른 평가를 받는 걸 구분하기 위함
2. 각 절을 형태소 분석 후, 속성 단어 앞뒤 윈도우(±4단어) 안에서 감성어 탐색
3. 감성어 강도(약/강)에 따라 기본 점수 부여, booster(정말/very)·dampener(살짝/slightly)로 가중치 조절, negation(안/not)으로 극성 반전
4. 리뷰×속성 단위 점수를 제품×속성 단위로 평균 집계
5. 최종 8개 속성으로 추려 대시보드 레이더차트용 데이터 생성
6. 제품별 긍정/부정 키워드 네트워크(HTML) 생성

### ⚠️ 참고 및 실행 전 준비사항
- 속성사전(`ASPECT_RULES`)과 감성어 강도 분류(`SENTI_WEAK/STRONG_POS/NEG`)를 만드는 탐색 과정(`show_freq`, `search_word`, `search_review`)은 생략하고, 최종 사전과 분석 로직만 정리했습니다.
- **최종 대시보드에는 정의된 속성 중 아래 8개만 사용했습니다.**
  - KR: 보습력, 발림성, 자극, 자외선차단, 백탁, 유지력, 가성비, 끈적임
  - GB: spreadability, white_cast, hydration, irritation, scent, price, sun_protection, lightweight
- 네트워크 시각화 결과(html)와 ABSA 결과 CSV는 리포지토리에 포함하지 않으며, 코드는 참고용으로만 공개합니다.
- 한국어 형태소 분석(KoNLPy Okt)은 Java(JDK) 설치가 필요하고, 영어 처리(NLTK)는 최초 1회 리소스 다운로드가 필요합니다.


## 1. 국내(KR) ABSA 분석

In [ ]:
#step 1. 라이브러리 호출
import pandas as pd
import re
from collections import Counter
from konlpy.tag import Okt

okt = Okt()

#step 2. 파일 불러오기 (경로는 실행 환경에 맞게 수정)
kr = pd.read_csv('kr_sentiment.csv')
kr = kr[kr['review_clean'].notna()].reset_index(drop=True)
print(f'전체: {len(kr):,}건')
print(f'긍정: {(kr["sentiment_label"]=="긍정").sum():,}건')
print(f'부정: {(kr["sentiment_label"]=="부정").sum():,}건')


In [ ]:
# 속성 사전 (반복 탐색 끝에 확정된 최종 버전)
ASPECT_RULES = {
    '발림성'    : {'발라', '발랏', '발랏는', '발랏는데', '발랏다', '발랏을때',
                   '발랏을땬', '발랴', '발리', '발리나', '발린', '발림', '발링', '흐르다', '묽다',
                   '흐르', '흐름'},
    '백탁'      : {'탁', '백탁현상', '백탁'},
    '끈적임'    : {'끈', '끈끈', '끈끈함', '끈덕', '끈덕끈', '끈쩍', '끈쩍끈쩍', '적임', '끈적끈적하다'},
    '보습력'    : {'촉촉', '수분', '보습', '건조'},
    '흡수력'    : {'흡수'},
    '유지력'    : {'유지', '유지됨', '지속'},
    '자극'      : {'화끈', '화끈거렸', '후끈'},
    '트러블'    : {'뽀로지', '뾰루지', '뽀루지', '여드름'},
    '수부지'    : {'수부지'},
    '복합성'    : {'복합성'},
    '지성'      : {'지성만', '지성은', '지성이면', '지성인'},
    '자외선차단': {'차단', '자외선'},
    '톤업'      : {'톤업', '톤', '보정', '커버', '베이스', '표현', '프리'},
    '향냄새'    : {'향', '냄새', '무향'},
    '무기자차'  : {'무기'},
    '유기자차'  : {'유기'},
    '가성비'    : {'가성', '저렴', '가격', '할인', '행사'},
    '밀착력'    : {'밀착', '밀착렫', '밀철'},
    '쿨링감'    : {'쿨롱', '쿨링', '쿨링금', '시원하다'},
    '구성상품'  : {'키링', '구성', '기획', '증정', '세트', '샘플', '사은'},
    '용량'      : {'양'},
    '잡티'      : {'흔적'},
    '유분기'    : {'기름기', '기름'}
}

norm_dict = {}
for target, sources in ASPECT_RULES.items():
    for source in sources:
        norm_dict[source] = target
print(f'속성 사전: {len(norm_dict)}개 항목')

# 감성어 정규화: 동의어/변형어를 대표 단어로 통일
SENTI_NORM = {
    '좋다'      : {'좋아하다', '굿', '굿굿'},
    '심하다'    : {'심하다', '심해', '심해지다'},
    '뜨다'      : {'들뜨다'},
    '밀리다'    : {'밀림'},
    '만족하다'  : {'만족스럽다', '만족', '만족도'},
    '싫어하다'  : {'싫다', '안좋다'},
    '추천하다'  : {'추천', '강추'},
    '투명하다'  : {'맑다'},
    '건조하다'  : {'당기다'},
    '뻑뻑하다'  : {'뻑뻑'},
    '시리다'    : {'시려웠', '시렵거', '시렵던', '시리', '시림'},
    '끼다'      : {'끼임'},
    '화사하다'  : {'하얘지다', '환하다', '밝아지다'},
    '부담스럽다': {'부담'}
}
senti_norm_dict = {}
for target, sources in SENTI_NORM.items():
    for source in sources:
        senti_norm_dict[source] = target

# 감성 강도 조절어
BOOSTERS = {'정말', '진짜', '완전', '아주', '거의', '전혀', '제일', '매우', '가장'}
DAMPENERS = {'좀', '약간', '살짝', '조금', '별로'}
NEGATIONS = {'안', '못', '않', '없', '아니', '않다', '없다', '못하다', '아니다'}

# 강도별 감성어 분류 (반복 탐색 끝에 확정된 최종 버전)
SENTI_WEAK_POS = {
    '무난', '괜찮다', '빠르다', '잘쓰다', '맞다', '가성비', '편하다', '많다', '얇다', '확실하다',
    '매끈하다', '귀엽다', '적당하다', '덜하다', '고르다', '은은하다', '투명하다', '안심',
    '넉넉하다', '놀라다', '충분하다', '착하다', '어울리다', '새롭다', '감사하다'
}

SENTI_STRONG_POS = {
    '좋다', '만족하다', '부드럽다', '산뜻하다',
    '촉촉하다', '순하다', '가볍다', '추천하다', '편안하다', '예쁘다', '자연스럽다', '깔끔하다',
    '필수', '딱이다', '저렴하다', '정착', '굉장하다', '잘맞다', '좋아지다', '이쁘다',
    '화사하다', '보송하다', '최고', '애용'
}

SENTI_WEAK_NEG = {
    '아쉽다', '뜨다', '올라오다', '심하다', '하얗다', '지워지다', '이상하다', '간지럽다'
}

SENTI_STRONG_NEG = {
    '밀리다', '따갑다', '답답하다', '부담스럽다', '무겁다',
    '끈적임', '뒤집어지다', '뻑뻑하다', '시리다', '버리다', '나쁘다',
    '싫어하다', '무너지다', '비싸다', '뭉치다', '건조하다', '아프다', '두껍다', '끼다',
    '힘들다', '불편하다'
}


In [ ]:
# 감성 점수 딕셔너리 (강도별 점수 부여)
SENTI_SCORES = {
    **{w: +0.8 for w in SENTI_STRONG_POS},
    **{w: +0.4 for w in SENTI_WEAK_POS},
    **{w: -0.4 for w in SENTI_WEAK_NEG},
    **{w: -0.8 for w in SENTI_STRONG_NEG},
}

BOOST_WEIGHT = 1.4
DAMP_WEIGHT  = 0.7
WINDOW       = 4

print(f'SENTI_SCORES 대상 단어 수: {len(SENTI_SCORES)}개')


In [ ]:
# 절 분리, 토크나이저, 점수 계산 함수
def split_clauses(text: str):
    """역접 접속사 기준으로 절 분리 (하지만/그런데/반면 등)"""
    t = re.sub(r'\s*(하지만|그런데|반면에|반면)\s*', r'|\1|', text)
    t = re.sub(r'(?:지만|인데|인데요|는데|는데요)(?=$|\s)', r'\g<0>|', t)
    parts = [p.strip() for p in t.split('|') if p.strip()]
    parts = [p for p in parts if p not in {'하지만', '그런데', '반면', '반면에'}]
    return parts if parts else [text]

def absa_tokenizer(text):
    """형태소 분석 + 속성/감성 정규화 (불용어 제거 없음 - window 내 booster/dampener/negation 감지 필요)"""
    morphs = okt.pos(str(text), stem=True, norm=True)
    tokens = []
    for word, pos in morphs:
        if pos in ['Noun', 'Adjective', 'Verb']:
            word = norm_dict.get(word, word)
            word = senti_norm_dict.get(word, word)
            tokens.append(word)
    return tokens

def calc_absa_score(clauses, aspect):
    """절별로 속성 단어 주변 윈도우 내 감성어를 찾아 점수 계산"""
    scores = []
    for tokens in clauses:
        if aspect not in tokens:
            continue
        for i, token in enumerate(tokens):
            if token != aspect:
                continue
            left  = max(0, i - WINDOW)
            right = min(len(tokens), i + WINDOW + 1)
            window_tokens = tokens[left:right]

            for j, w in enumerate(window_tokens):
                base = SENTI_SCORES.get(w, 0)
                if base == 0:
                    continue
                weight = 1.0
                local = window_tokens[max(0, j-2):j+3]

                if any(b in local for b in BOOSTERS):
                    weight *= BOOST_WEIGHT
                if any(d in local for d in DAMPENERS):
                    weight *= DAMP_WEIGHT

                neg_window = window_tokens[max(0, j-3):j+4]
                if any(n in neg_window for n in NEGATIONS):
                    base *= -1

                scores.append(base * weight)

    return round(sum(scores) / len(scores), 3) if scores else None


In [ ]:
# 형태소 분석 미리 실행 (속도 개선) - 절 분리 → 각 절 토크나이징 → clauses 컬럼으로 저장
print('형태소 분석 중... (시간이 걸려요)')
kr['clauses'] = kr['review_clean'].apply(
    lambda x: [absa_tokenizer(c) for c in split_clauses(str(x))]
)
print('형태소 분석 완료!')


In [ ]:
# ABSA 실행: 리뷰별 × 속성별 감성 점수 계산
aspects = list(ASPECT_RULES.keys())
results = []
total = len(kr)

print('ABSA 분석 중...')
for idx, row in kr.iterrows():
    if idx % 500 == 0:
        print(f'  {idx:,} / {total:,}건 처리중...')

    clauses = row['clauses']
    product = row.get('product_name', '')

    for aspect in aspects:
        score = calc_absa_score(clauses, aspect)
        if score is None:
            continue
        label = '긍정' if score > 0 else '부정' if score < 0 else '중립'
        results.append({
            'product_name': product,
            'aspect': aspect,
            'score': score,
            'label': label,
        })

df_absa = pd.DataFrame(results)
print(f'완료: {len(df_absa):,}개 속성-리뷰 쌍')


In [ ]:
# 제품별 × 속성별 평균 점수 집계
df_product = (
    df_absa
    .groupby(['product_name', 'aspect'])['score']
    .agg(['mean', 'count'])
    .reset_index()
    .rename(columns={'mean': 'avg_score', 'count': 'review_count'})
)
df_product['avg_score'] = df_product['avg_score'].round(3)
df_product['label'] = df_product['avg_score'].apply(
    lambda x: '긍정' if x > 0.1 else '부정' if x < -0.1 else '중립'
)
# 리뷰 5건 미만 속성은 신뢰도 낮으므로 제외
df_product = df_product[df_product['review_count'] >= 5]

# 결과 저장 (경로는 실행 환경에 맞게 수정 - 결과 파일은 리포지토리에는 포함하지 않음)
df_absa.to_csv('absa_results.csv', index=False, encoding='utf-8-sig')
df_product.to_csv('absa_product_summary.csv', index=False, encoding='utf-8-sig')
print(f'저장 완료: absa_results.csv ({len(df_absa):,}행), absa_product_summary.csv ({len(df_product):,}행)')


In [ ]:
# 대시보드(app_final.py)에서 실제 사용 중인 최종 8개 속성
KR_ASPECTS = ['보습력', '발림성', '자극', '자외선차단', '백탁', '유지력', '가성비', '끈적임']
print(f'KR 최종 속성: {KR_ASPECTS}')


### 제품별 키워드 네트워크 시각화 (KR)

In [ ]:
import os
import networkx as nx
from pyvis.network import Network

net_base = './network_analysis_kr'
os.makedirs(net_base, exist_ok=True)
aspect_values = set(ASPECT_RULES.keys())

def get_senti_tokens_by_product(product_name):
    """제품별 긍/부정 감성어 + 속성어 추출"""
    df_p = kr[kr['product_name'] == product_name]
    pos_by_doc, neg_by_doc = [], []

    for _, row in df_p.iterrows():
        tokens = [t for clause in row['clauses'] for t in clause]
        pos_words = list(set(t for t in tokens if SENTI_SCORES.get(t, 0) > 0 or t in aspect_values))
        neg_words = list(set(t for t in tokens if SENTI_SCORES.get(t, 0) < 0 or t in aspect_values))
        if pos_words:
            pos_by_doc.append(pos_words)
        if neg_words:
            neg_by_doc.append(neg_words)
    return pos_by_doc, neg_by_doc

def build_network_html(docs, node_color, tmp_path, top_n=12, min_edge=2):
    """공출현 기반 pyvis 네트워크 HTML 생성"""
    node_cnt, edge_cnt = Counter(), Counter()
    for words in docs:
        node_cnt.update(words)
        for i in range(len(words)):
            for j in range(i+1, len(words)):
                a, b = sorted([words[i], words[j]])
                edge_cnt[(a, b)] += 1

    if not node_cnt:
        return False

    top_nodes = {w for w, _ in node_cnt.most_common(top_n)}
    G = nx.Graph()
    for w in top_nodes:
        G.add_node(w, freq=node_cnt[w])
    for (a, b), cnt in edge_cnt.items():
        if cnt >= min_edge and a in top_nodes and b in top_nodes:
            G.add_edge(a, b, weight=cnt)

    if G.number_of_edges() == 0:
        return False

    net = Network(height='650px', width='100%', directed=False,
                  bgcolor='#2d2d2d', font_color='#ffffff')
    net.barnes_hut(gravity=-10000, spring_length=160, spring_strength=0.02, damping=0.6)

    max_freq = max(node_cnt.values())
    for w in G.nodes():
        freq = node_cnt[w]
        size = 10 + 35 * (freq / max_freq)
        net.add_node(w, label=w, size=round(size, 1), color=node_color,
                     title=f'{w}<br>등장: {freq}회',
                     font={'color': '#ffffff', 'size': 15, 'bold': True, 'face': 'Malgun Gothic'})

    weights = nx.get_edge_attributes(G, 'weight')
    max_w = max(weights.values()) if weights else 1
    for u, v, data in G.edges(data=True):
        w = data.get('weight', 1)
        net.add_edge(u, v, value=round(1 + 5 * (w / max_w), 2),
                     title=f'{u}—{v} {w}회', color='rgba(255,255,255,0.2)')

    net.write_html(tmp_path, open_browser=False)
    return True

def build_dual_html(product_name, out_html, top_n=12, min_edge=2):
    """긍정/부정 네트워크를 나란히 배치한 HTML 파일 생성"""
    pos_docs, neg_docs = get_senti_tokens_by_product(product_name)
    print(f'  긍정: {len(pos_docs)}건 / 부정: {len(neg_docs)}건')

    tmp_pos = out_html.replace('.html', '_tmp_pos.html')
    tmp_neg = out_html.replace('.html', '_tmp_neg.html')
    has_pos = build_network_html(pos_docs, '#5BB8E6', tmp_pos, top_n, min_edge)
    has_neg = build_network_html(neg_docs, '#E87090', tmp_neg, top_n, min_edge)
    pos_src, neg_src = os.path.basename(tmp_pos), os.path.basename(tmp_neg)

    html = f"""<!DOCTYPE html><html><head><meta charset="utf-8">
<style>
* {{ margin:0; padding:0; box-sizing:border-box; }}
body {{ font-family:'Malgun Gothic',sans-serif; background:#1e1e1e; }}
h1 {{ text-align:center; padding:14px; font-size:15px; color:#fff; background:#2d2d2d; border-bottom:1px solid #444; }}
.wrap {{ display:grid; grid-template-columns:1fr 1fr; gap:12px; padding:12px; height:calc(100vh - 52px); }}
.card {{ background:#2d2d2d; border-radius:10px; border:1px solid #444; display:flex; flex-direction:column; overflow:hidden; }}
.card-title {{ padding:10px 16px; font-size:13px; font-weight:bold; }}
.pos-title {{ color:#5BB8E6; background:#1a3a4a; }}
.neg-title {{ color:#E87090; background:#4a1a2a; }}
iframe {{ flex:1; border:none; width:100%; height:650px; }}
</style></head><body>
<h1>{product_name} — 리뷰 키워드 네트워크</h1>
<div class="wrap">
  <div class="card"><div class="card-title pos-title">긍정 키워드 네트워크</div>
    {f'<iframe src="{pos_src}"></iframe>' if has_pos else '<div style="padding:20px;color:#999;text-align:center">데이터 없음</div>'}
  </div>
  <div class="card"><div class="card-title neg-title">부정 키워드 네트워크</div>
    {f'<iframe src="{neg_src}"></iframe>' if has_neg else '<div style="padding:20px;color:#999;text-align:center">데이터 없음</div>'}
  </div>
</div></body></html>"""

    with open(out_html, 'w', encoding='utf-8') as f:
        f.write(html)
    print(f'  저장: {out_html}')


In [ ]:
# 제품별 네트워크 HTML 생성 (예시 - 나머지 상품 생략)
order = ['1_product_a', '2_product_b']

print('네트워크 생성 중...')
for product in order:
    print(f'\n[{product}]')
    build_dual_html(
        product_name=product,
        out_html=f'{net_base}/{product}_network.html',
        top_n=12, min_edge=2
    )
print('=== 전체 완료 ===')


## 2. 글로벌(GB) ABSA 분석

In [ ]:
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag, word_tokenize
lemmatizer = WordNetLemmatizer()

# 파일 불러오기 (경로는 실행 환경에 맞게 수정)
gb = pd.read_csv('gb_sentiment.csv')
gb = gb[gb['review_clean'].notna()].reset_index(drop=True)
print(f'전체: {len(gb):,}건')
print(f'긍정: {(gb["sentiment_label"]=="긍정").sum():,}건')
print(f'부정: {(gb["sentiment_label"]=="부정").sum():,}건')


In [ ]:
# 속성 사전 (반복 탐색 끝에 확정된 최종 버전)
ASPECT_RULES_GB = {
    'white_cast'        : {'white', 'cast', 'whitecast', 'casting'},
    'lightweight'       : {'light', 'lightweighted', 'litghweight', 'weightless'},
    'sticky'            : {'stickiness', 'tacky', 'stick', 'stickey', 'stickness', 'residue'},
    'scent'             : {'smell', 'fragrance'},
    'sun_protection'    : {'protection', 'sunprotection', 'spf', 'protect', 'uv', 'protects'},
    'absorption'        : {'absorb', 'absorbance', 'absorbed', 'absorbency', 'absorbent',
                            'absorber', 'absorbes', 'absorbing', 'absorbs'},
    'hydration'         : {'hydrate', 'hydrated', 'hydrating', 'hydrator', 'hydrators',
                            'moist', 'moisturizing', 'moisturize', 'moisturizes', 'moisturise'},
    'glowy'             : {'glow', 'glowy', 'shiny', 'shine', 'glowing'},
    'irritation'        : {'irritate', 'irritated', 'irritant', 'irritates', 'irritating', 'redness'},
    'acne'              : {'breakout', 'pimple', 'blemish', 'prone', 'breakouts'},
    'watery'            : {'runny', 'liquid'},
    'oily'              : {'oil', 'oiler', 'oiliness', 'oily', 'greasiness', 'grease'},
    'price'             : {'price', 'deal', 'sale', 'value', 'worth'},
    'consistency'       : {'last', 'long'},
    'mineral_sunscreen' : {'mineral'},
    'chemical_sunscreen': {'chemical'},
    'pilling'           : {'pill', 'pilling', 'pilled', 'pills'},
    'delivery'          : {'ship', 'shipment', 'shipping', 'deliver', 'delivers', 'arrive'},
    'eye_irritation'    : {'eye', 'eyes'},
    'brightening'       : {'brighten', 'bright', 'brightens', 'brighter', 'brighting', 'brightness'},
    'spreadability'     : {'spread', 'blend', 'rub', 'layer', 'blendable', 'blending', 'spreadable'},
    'texture'           : {'texture', 'formula'},
    'finish'            : {'finish', 'dewy', 'matte'},
    'amount'            : {'amount'},
    'coverage'          : {'coverage', 'cover'},
    'pore'              : {'pores'},
    'normal_skintype'       : {'normal'},
    'combination_skintype'  : {'comnbination'},
    'sensitive_skintype'    : {'sensitive'}
}

norm_dict_gb = {}
for target, sources in ASPECT_RULES_GB.items():
    for source in sources:
        norm_dict_gb[source] = target
print(f'속성 수: {len(ASPECT_RULES_GB)}개 / 속성 사전 항목: {len(norm_dict_gb)}개')

# 감성어 정규화: 동의어/변형어를 대표 단어로 통일
SENTI_NORM_GB = {
    'heavy'      : {'thick'},
    'favorite'   : {'fav', 'fave', 'faves', 'favorit', 'favorite', 'favorites', 'favortie', 'grail', 'holy'},
    'recommend'  : {'recom', 'recomend', 'recommendation', 'recommends', 'recommended'},
    'calm'       : {'calmer', 'calming', 'calms'},
    'repurchase' : {'repeat'},
    'clog'       : {'cloggers', 'clogging'},
    'soothing'   : {'soothe'}
}
senti_norm_dict_gb = {}
for target, sources in SENTI_NORM_GB.items():
    for source in sources:
        senti_norm_dict_gb[source] = target

# 감성 강도 조절어
BOOSTERS_GB  = {'very', 'really', 'extremely', 'super', 'much', 'lot', 'more', 'strong', 'many', 'high', 'absolute'}
DAMPENERS_GB = {'slightly', 'quite', 'little', 'bit', 'few', 'slight', 'less'}
NEGATIONS_GB = {'not', "n't", 'no', 'never', 'neither', 'nor', 'non',
                'without', 'lack', 'fail', 'doesn', 'didn', 'don', 'isn', 'doesnt', 'wasn'}

# 감성어 강도별 분류
SENTI_WEAK_POS_GB = {
    'nice', 'comfortable', 'suitable', 'gentle', 'soft', 'fresh', 'natural', 'help',
    'easy', 'calm', 'fast', 'thin', 'right', 'fine', 'effect', 'healty', 'creamy', 'refresh',
    'reliable', 'fit', 'quick', 'convenient', 'soothing', 'cheap'
}
SENTI_STRONG_POS_GB = {
    'good', 'love', 'recommend', 'smooth', 'great', 'best', 'favorite', 'perfect', 'repurchase', 'excellent',
    'amaze', 'amazing', 'beautiful', 'favourite', 'happy', 'glad', 'pleasant', 'dealbreaker', 'enjoy',
    'effective', 'satisfied'
}
SENTI_WEAK_NEG_GB = {'disappoint', 'heavy', 'break', 'hard', 'issue', 'oily'}
SENTI_STRONG_NEG_GB = {'bad', 'waste', 'expensive', 'dry', 'greasy', 'burn', 'pilling', 'sting', 'clog'}


In [ ]:
# 감성 점수 딕셔너리
SENTI_SCORES_GB = {
    **{w: +0.8 for w in SENTI_STRONG_POS_GB},
    **{w: +0.4 for w in SENTI_WEAK_POS_GB},
    **{w: -0.4 for w in SENTI_WEAK_NEG_GB},
    **{w: -0.8 for w in SENTI_STRONG_NEG_GB},
}
BOOST_WEIGHT_GB = 1.4
DAMP_WEIGHT_GB  = 0.7
WINDOW_GB       = 4
print(f'SENTI_SCORES 대상 단어 수: {len(SENTI_SCORES_GB)}개')

def get_wordnet_pos(tag):
    from nltk.corpus import wordnet
    if tag.startswith('NN'): return wordnet.NOUN
    if tag.startswith('JJ'): return wordnet.ADJ
    if tag.startswith('VB'): return wordnet.VERB
    return None

def split_clauses_gb(text: str):
    """역접 접속사 기준으로 절 분리 (but/however 등)"""
    t = re.sub(r'\s*(but|however|although|though|yet|while|whereas)\s*',
               r'|\1|', text, flags=re.IGNORECASE)
    parts = [p.strip() for p in t.split('|') if p.strip()]
    connectors = {'but', 'however', 'although', 'though', 'yet', 'while', 'whereas'}
    parts = [p for p in parts if p.lower() not in connectors]
    return parts if parts else [text]

def absa_tokenizer_gb(text):
    """ABSA 점수 계산용 토크나이저 (불용어 제거 없음 - window 내 booster/dampener/negation 감지 필요)"""
    tokens_raw = word_tokenize(str(text).lower())
    tagged = pos_tag(tokens_raw)
    tokens = []
    for word, tag in tagged:
        wn_pos = get_wordnet_pos(tag)
        if wn_pos is None:
            if word in BOOSTERS_GB or word in DAMPENERS_GB or word in NEGATIONS_GB:
                tokens.append(word)
            continue
        if len(word) < 2:
            continue
        lemma = lemmatizer.lemmatize(word, pos=wn_pos)
        lemma = norm_dict_gb.get(lemma, lemma)
        lemma = senti_norm_dict_gb.get(lemma, lemma)
        tokens.append(lemma)
    return tokens

def calc_absa_score_gb(clauses, aspect):
    """절별로 속성 단어 주변 윈도우 내 감성어를 찾아 점수 계산"""
    scores = []
    for tokens in clauses:
        if aspect not in tokens:
            continue
        for i, token in enumerate(tokens):
            if token != aspect:
                continue
            left = max(0, i - WINDOW_GB)
            right = min(len(tokens), i + WINDOW_GB + 1)
            window_tokens = tokens[left:right]

            for j, w in enumerate(window_tokens):
                base = SENTI_SCORES_GB.get(w, 0)
                if base == 0:
                    continue
                weight = 1.0
                local = window_tokens[max(0, j-2):j+3]
                neg_window = window_tokens[max(0, j-3):j+4]

                if any(b in local for b in BOOSTERS_GB):
                    weight *= BOOST_WEIGHT_GB
                if any(d in local for d in DAMPENERS_GB):
                    weight *= DAMP_WEIGHT_GB
                if any(n in neg_window for n in NEGATIONS_GB):
                    base *= -1

                scores.append(base * weight)
    return round(sum(scores) / len(scores), 3) if scores else None


In [ ]:
# 형태소 분석 미리 실행 (속도 개선)
print('형태소 분석 중... (시간이 걸려요)')
gb['clauses'] = gb['review_clean'].apply(
    lambda x: [absa_tokenizer_gb(c) for c in split_clauses_gb(str(x))]
)
print('형태소 분석 완료!')


In [ ]:
# ABSA 실행: 리뷰별 × 속성별 감성 점수 계산
aspects_gb = list(ASPECT_RULES_GB.keys())
results_gb = []
total = len(gb)

print('ABSA 분석 중...')
for idx, row in gb.iterrows():
    if idx % 500 == 0:
        print(f'  {idx:,} / {total:,}건 처리중...')

    for aspect in aspects_gb:
        score = calc_absa_score_gb(row['clauses'], aspect)
        if score is None:
            continue
        label = '긍정' if score > 0 else '부정' if score < 0 else '중립'
        results_gb.append({
            'product_name': row.get('product_name', ''),
            'aspect': aspect,
            'score': score,
            'label': label,
        })

df_absa_gb = pd.DataFrame(results_gb)
print(f'완료: {len(df_absa_gb):,}개 속성-리뷰 쌍')


In [ ]:
# 제품별 × 속성별 평균 점수 집계 + 저장
df_product_gb = (
    df_absa_gb
    .groupby(['product_name', 'aspect'])['score']
    .agg(['mean', 'count'])
    .reset_index()
    .rename(columns={'mean': 'avg_score', 'count': 'review_count'})
)
df_product_gb['avg_score'] = df_product_gb['avg_score'].round(3)
df_product_gb['label'] = df_product_gb['avg_score'].apply(
    lambda x: '긍정' if x > 0.1 else '부정' if x < -0.1 else '중립'
)
df_product_gb = df_product_gb[df_product_gb['review_count'] >= 5]

df_absa_gb.to_csv('absa_results_gb.csv', index=False, encoding='utf-8-sig')
df_product_gb.to_csv('absa_product_summary_gb.csv', index=False, encoding='utf-8-sig')
print(f'저장 완료: absa_results_gb.csv ({len(df_absa_gb):,}행), absa_product_summary_gb.csv ({len(df_product_gb):,}행)')


In [ ]:
# 대시보드(app_final.py)에서 실제 사용 중인 최종 8개 속성
GB_ASPECTS = ['spreadability', 'white_cast', 'hydration',
              'irritation', 'scent', 'price',
              'sun_protection', 'lightweight']
print(f'GB 최종 속성: {GB_ASPECTS}')


### 제품별 키워드 네트워크 시각화 (GB)

In [ ]:
net_base_gb = './network_analysis_gb'
os.makedirs(net_base_gb, exist_ok=True)
aspect_values_gb = set(ASPECT_RULES_GB.keys())

def get_senti_tokens_by_product_gb(product_name):
    """제품별 긍/부정 감성어 + 속성어 추출"""
    df_p = gb[gb['product_name'] == product_name]
    pos_by_doc, neg_by_doc = [], []

    for _, row in df_p.iterrows():
        tokens = [t for clause in row['clauses'] for t in clause]
        pos_words = list(set(t for t in tokens if SENTI_SCORES_GB.get(t, 0) > 0 or t in aspect_values_gb))
        neg_words = list(set(t for t in tokens if SENTI_SCORES_GB.get(t, 0) < 0 or t in aspect_values_gb))
        if pos_words:
            pos_by_doc.append(pos_words)
        if neg_words:
            neg_by_doc.append(neg_words)
    return pos_by_doc, neg_by_doc

# 네트워크 생성 함수(build_network_html, build_dual_html)는 KR 섹션과 동일한 함수를 그대로 사용합니다.


In [ ]:
# 제품별 네트워크 HTML 생성 (예시 - 나머지 상품 생략)
order_gb = ['1_product_a', '2_product_b']

print('네트워크 생성 중...')
for product in order_gb:
    print(f'\n[{product}]')
    pos_docs, neg_docs = get_senti_tokens_by_product_gb(product)
    print(f'  긍정: {len(pos_docs)}건 / 부정: {len(neg_docs)}건')

    tmp_pos = f'{net_base_gb}/{product}_tmp_pos.html'
    tmp_neg = f'{net_base_gb}/{product}_tmp_neg.html'
    build_network_html(pos_docs, '#5BB8E6', tmp_pos, top_n=12, min_edge=2)
    build_network_html(neg_docs, '#E87090', tmp_neg, top_n=12, min_edge=2)

print('=== 전체 완료 ===')
